# 1 DATA PREPARATION

**Project:** Irish Company Survival: Registrations, Cohort Survival and Prediction

**Author:** Temuulen Bulgan

### Purpose of this notebook
This notebook loads the raw company register, checks its quality and turns it into one clean table of companies that the rest of the project uses for analysis, SQL and the Power BI dashboard.

### Data sources
| Dataset | Source | Content |
|---|---|---|
| Company Records (`companies.csv.zip`) | Companies Registration Office (CRO) | One row per company registered in Ireland, including dissolved companies: registration date, status, dissolution date, company type, sector code and address |

### Contents
1. Company Records: inspection
2. Company Records: cleaning
3. Saving the clean data

### Output
This notebook saves the cleaned data to `data/processed/companies_clean.csv` (one row per company).

## Setup

In [1]:
import pandas as pd
from pathlib import Path

# Show all columns when displaying a DataFrame
pd.set_option("display.max_columns", None)

## 1. Company Records: inspection

**Source:** Companies Registration Office (CRO), dataset "Company Records". Downloaded from opendata.cro.ie in October 2026. Licence: CC BY 4.0. Contains Irish Public Sector Data licensed under a Creative Commons Attribution 4.0 International (CC BY 4.0) licence.

**Structure according to the source:** one row per company registered in Ireland, including dissolved companies, with registration date, status, dissolution date, company type, sector codes and address.

In this section I inspect the raw file before cleaning it.

### 1.1 Inspection

#### Load the file and check its size

In [2]:
# Load the raw file
companies = pd.read_csv("../data/raw/companies.csv.zip")

# Number of rows and columns
companies.shape

(827902, 21)

The file has 827,902 rows (one per company) and 21 columns.

#### First and last rows

In [3]:
# First and last 3 rows: format of the data, and where it starts and ends
display(companies.head(3))
display(companies.tail(3))

,company_num,company_name,company_status_code,company_status,company_type_code,company_type,company_reg_date,last_ar_date,company_address_1,company_address_2,company_address_3,company_address_4,comp_dissolved_date,nard,last_accounts_date,company_status_date,nace_v2_code,eircode,company_name_eff_date,company_type_eff_date,princ_object_code
0,340217,STORM ENTERPRISES LIMITED,1151,Normal,1153,LTD - Private Company Limited by Shares,2001-03-13,2025-09-30,C/O Storm Technology,The Capel Building,Marys Abbey,",DUBLIN",NaN,2026-09-30,2024-12-31,NaN,NaN,D07 N977,2015-11-05,2016-12-01,72.20
1,340235,L.M.G. DEVELOPMENTS LIMITED,1158,Dissolved,1153,LTD - Private Company Limited by Shares,2001-03-13,2016-09-13,7 GLEBE CREST,DONEGAL TOWN,"CO. DONEGAL,DONEGAL","CO. DONEGAL, DONEGAL, IRELAND",2017-11-29,2017-09-13,2016-03-31,2017-11-29,NaN,F94H4X5,2001-03-13,2016-12-01,45.21
2,340241,M V FOODS LIMITED,1151,Normal,1153,LTD - Private Company Limited by Shares,2001-03-13,2025-12-31,UNIT 1 CLANBRASSIL CENTRE,CLANBRASSIL STREET,DUNDALK CO.LOUTH.,"DUNDALK,LOUTH",NaN,2026-12-31,2025-03-31,NaN,NaN,NaN,2001-03-13,2016-12-01,52.22


,company_num,company_name,company_status_code,company_status,company_type_code,company_type,company_reg_date,last_ar_date,company_address_1,company_address_2,company_address_3,company_address_4,comp_dissolved_date,nard,last_accounts_date,company_status_date,nace_v2_code,eircode,company_name_eff_date,company_type_eff_date,princ_object_code
827899,829192,CK CASTLE PROPERTY SERVICES LIMITED,1151,Normal,1153,LTD - Private Company Limited by Shares,2026-10-27,NaN,82 PARK STREET,NaN,NaN,"DUNDALK,LOUTH",NaN,2027-04-27,NaN,NaN,5630.0,A91EC85,2026-10-27,2026-10-27,NaN
827900,829212,ANGEL GLOBAL VOYAGER LIMITED,1151,Normal,1153,LTD - Private Company Limited by Shares,2026-10-27,NaN,THE BLACK CHURCH,ST. MARY'S PLACE,NaN,DUBLIN 7,NaN,2027-04-27,NaN,NaN,5829.0,D07P4AX,2026-10-27,2026-10-27,NaN
827901,829189,SLM DENTAL CARE LIMITED,1151,Normal,1153,LTD - Private Company Limited by Shares,2026-10-27,NaN,Birch Lane,Iskaheen,"Muff,DONEGAL",NaN,NaN,2027-04-27,NaN,NaN,8623.0,F93 RD83,2026-10-27,2026-10-27,NaN


First look: there is one row per company. The file does not start with the oldest companies: the first rows were registered on 13 March 2001 and the last rows on 27 October 2026. The company numbers are not in strict order either. The 2001 rows have a sector code in `princ_object_code` but not in `nace_v2_code`, and the 2026 rows have the opposite. `nace_v2_code` is shown with a decimal point (for example `5630.0`). Some text has extra spaces or mixed formats, for example the addresses (`,DUBLIN`, `CO. DONEGAL,DONEGAL`).

#### Column types and missing counts

In [4]:
# Column names, data types and non-missing counts
companies.info()

<class 'pandas.DataFrame'>
RangeIndex: 827902 entries, 0 to 827901
Data columns (total 21 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   company_num            827902 non-null  int64  
 1   company_name           827902 non-null  str    
 2   company_status_code    827902 non-null  int64  
 3   company_status         827902 non-null  str    
 4   company_type_code      827902 non-null  int64  
 5   company_type           827902 non-null  str    
 6   company_reg_date       811710 non-null  str    
 7   last_ar_date           610066 non-null  str    
 8   company_address_1      827902 non-null  str    
 9   company_address_2      766947 non-null  str    
 10  company_address_3      572718 non-null  str    
 11  company_address_4      802369 non-null  str    
 12  comp_dissolved_date    475006 non-null  str    
 13  nard                   802906 non-null  str    
 14  last_accounts_date     501102 non-null  str    

The identifier, name, status and type columns are complete (827,902 non-null). `company_reg_date` has fewer values (811,710), so some companies have no registration date. This matters because every survival measure starts from this date. All date columns are stored as text (`str`), so they need converting to dates. `nace_v2_code` is stored as a number (`float64`), which explains the `.0` seen earlier. It should be text, because it is a code, not an amount. Both sector code columns and `comp_dissolved_date` are only partly filled. The next check counts the missing values in each column.

#### Missing values per column

In [5]:
# Number of missing values in each column
missing_count = companies.isna().sum()

# Missing values as a percentage of all rows
missing_pct = (missing_count / len(companies) * 100).round(1)

# Put both in one table, largest first
missing = pd.DataFrame({"missing": missing_count, "missing_pct": missing_pct})
missing.sort_values("missing", ascending=False)

,missing,missing_pct
nace_v2_code,561909,67.9
eircode,533605,64.5
princ_object_code,369827,44.7
comp_dissolved_date,352896,42.6
last_accounts_date,326800,39.5
company_status_date,320925,38.8
company_address_3,255184,30.8
last_ar_date,217836,26.3
company_address_2,60955,7.4
company_address_4,25533,3.1


The largest gaps are in the sector codes (`nace_v2_code` about 68% missing, `princ_object_code` about 45%) and `eircode` (about 65%). `comp_dissolved_date` is missing for about 43% of companies, which is expected if most of these are companies that are still registered. A later check confirms this against `company_status`. `company_reg_date` is missing for 16,192 companies (2.0%). The name and type effective dates have almost the same number of gaps, so these may be the same companies. Seven columns have no missing values.

#### Company status categories
This column decides which companies count as closed, so I list every value and how often it appears.

In [6]:
# Every status value and its number of companies (including missing)
companies["company_status"].value_counts(dropna=False)

company_status
Dissolved                           458801
Normal                              323502
Dissolved-20 years                   17653
Strike Off Listed                    10806
Liquidation                           9211
Ceased IRL                            2676
Dissolved PostMerger                  2100
Ceased                                1254
Normal                                 900
Struck Off                             687
Deleted CB Merger                      182
De-Registered ICAV                      68
Conv. to IndProvSoc                     22
Administration (UK)                     12
Deleted CB Conversion                   10
Deleted per SE merge                     7
Deleted as per SE Tr                     3
Examinership                             2
Liquidation (UK)                         2
Administration                           1
Liquidation stayed                       1
De-Registered Investment Company         1
Dissolved by Divis                     

#### Hidden spaces in status values
`Normal` appears twice in the table above. Here I show each value in quotes, so that spaces at the start or end are visible.m

In [7]:
# Each status value in quotes, with its number of companies
status_counts = companies["company_status"].value_counts()

for status, count in status_counts.items():
    print(repr(status), count)

'Dissolved' 458801
'Normal ' 323502
'Dissolved-20 years' 17653
'Strike Off Listed' 10806
'Liquidation' 9211
'Ceased IRL' 2676
'Dissolved PostMerger' 2100
'Ceased' 1254
'Normal' 900
'Struck Off' 687
'Deleted CB Merger' 182
'De-Registered ICAV' 68
'Conv. to IndProvSoc' 22
'Administration (UK)' 12
'Deleted CB Conversion' 10
'Deleted per SE merge' 7
'Deleted as per SE Tr' 3
'Examinership' 2
'Liquidation (UK)' 2
'Administration' 1
'Liquidation stayed' 1
'De-Registered Investment Company' 1
'Dissolved by Divis' 1


There are 23 status values. The two largest are `Dissolved` and `Normal `. `Normal` appears twice because 323,502 values have a trailing space and 900 do not. These are the same status, so cleaning will remove the spaces before grouping. Besides these, there are dissolved variants (`Dissolved-20 years`, `Dissolved PostMerger`), statuses for companies being closed (`Strike Off Listed`, `Liquidation`) and many small groups with fewer than 200 companies each. A rule is needed for which of these count as closed.

#### Company type categories
Company type is one of the features for the survival model, so I check its values.

In [8]:
# Number of different company types
print("Number of company types:", companies["company_type"].nunique())

# Every company type and its number of companies
companies["company_type"].value_counts()

Number of company types: 44


company_type
LTD - Private Company Limited by Shares                                        405121
Private limited by shares                                                      240139
Private                                                                         49224
Single member private company limited by shares                                 45474
CLG - Company Limited by Guarantee                                              22459
Unknown                                                                         15290
DAC - Designated Activity Company (limited by shares)                           11152
ULC - Private Unlimited Company                                                  7903
External company                                                                 7819
Private unlimited with share capital                                             7208
Guarantee company without a share capital (public)                               6867
Private unlimited                        

There are 44 company types. The same type appears under different labels: for example `LTD - Private Company Limited by Shares` (405,121) and `Private limited by shares` (240,139), or `PLC - Public Limited Company` and `Public limited company`. One label system uses short codes such as `LTD`, `CLG` and `DAC`, and the other uses longer descriptions. 15,290 companies have the type `Unknown`. Many types have fewer than 100 companies. For the model, the types will need grouping into a few main groups.

#### Company type by status
To see why the same type has two labels, I compare each of the 10 largest types with whether the company is still `Normal` (active).

In [9]:
# True when the status is Normal (spaces removed only for this comparison)
is_normal = companies["company_status"].str.strip() == "Normal"

# Number of companies per type, split by Normal (True) or not (False)
type_by_status = pd.crosstab(companies["company_type"], is_normal)

# Show the 10 largest types
top_types = companies["company_type"].value_counts().head(10).index
type_by_status.loc[top_types]

company_status,False,True
company_type,,
LTD - Private Company Limited by Shares,117805,287316
Private limited by shares,240091,48
Private,49223,1
Single member private company limited by shares,45459,15
CLG - Company Limited by Guarantee,4053,18406
Unknown,15290,0
DAC - Designated Activity Company (limited by shares),3891,7261
ULC - Private Unlimited Company,2622,5281
External company,4184,3635


The two label systems split by status. The long labels (`Private limited by shares`, `Private`, `Single member private company limited by shares`, `Private unlimited with share capital`) belong almost only to companies that are no longer `Normal`. The short-code labels (`LTD`, `CLG`, `DAC`, `ULC`) include most of the active companies. This suggests that the short labels are the current types and that closed companies kept their old labels. All 15,290 `Unknown` companies are not `Normal`. When cleaning, old and new labels of the same type will be mapped to one group.

#### Registration dates
The registration date is stored as text. I convert a copy to dates to check that every value is a valid date and to see the earliest and latest dates.

In [10]:
# Convert a copy of the column to dates; values that are not valid dates become missing (NaT)
reg_date = pd.to_datetime(companies["company_reg_date"], format="%Y-%m-%d", errors="coerce")

# Text values that could not be converted (filled in the raw column, missing after converting)
not_converted = companies["company_reg_date"].notna() & reg_date.isna()
print("Missing in raw file:", companies["company_reg_date"].isna().sum())
print("Could not be converted:", not_converted.sum())

# Earliest and latest registration date
print("Earliest:", reg_date.min())
print("Latest:", reg_date.max())

Missing in raw file: 16192
Could not be converted: 0
Earliest: 1753-01-01 00:00:00
Latest: 2026-10-27 00:00:00


All filled registration dates are valid dates in the same format, and 16,192 are missing. The latest date is 27 October 2026, the day before the download. The earliest date, 1 January 1753, is not a realistic registration date and looks like a placeholder. The next check looks at the oldest dates.

#### The oldest registration dates

In [11]:
# Number of companies with each of the oldest registration dates
print("Registered in 1753:", (reg_date.dt.year == 1753).sum())
print("Registered before 1900:", (reg_date < "1900-01-01").sum())

# The company registered in 1753
companies[reg_date.dt.year == 1753].T

Registered in 1753: 1
Registered before 1900: 157


,54248
company_num,960000
company_name,FORESIGHT HOLDINGS LIMITED
company_status_code,1151
company_status,Normal
company_type_code,1150
company_type,DAC - Designated Activity Company (limited by ...
company_reg_date,1753-01-01
last_ar_date,NaN
company_address_1,********NO ADDRESS DETAILS*******
company_address_2,********NO ADDRESS DETAILS*******


Only one company has the date 1753. Its record has no address details and several other unrealistic dates, so the 1753 date is a placeholder, not a real registration date. 157 companies were registered before 1900. The next check compares older and newer registration periods, to decide which years the analysis should use.

#### Registration periods: choosing the start year
Here I group companies by 5-year registration period and check two things for each period: how many companies have the status `Dissolved-20 years`, and how many have a sector code.

In [12]:
# 5-year registration period (for example 2003 becomes 2000)
period = (reg_date.dt.year // 5) * 5

# Companies with the status Dissolved-20 years, and companies with any sector code
is_dissolved_20y = companies["company_status"].str.strip() == "Dissolved-20 years"
has_sector = companies["nace_v2_code"].notna() | companies["princ_object_code"].notna()

# Count, Dissolved-20 years and % with a sector code, per period
period_summary = pd.DataFrame({
    "companies": period.value_counts(),
    "dissolved_20y": is_dissolved_20y.groupby(period).sum(),
    "sector_pct": (has_sector.groupby(period).mean() * 100).round(1),
}).sort_index()

period_summary[period_summary.index >= 1960]

,companies,dissolved_20y,sector_pct
company_reg_date,,,
1960.0,2406,118,10.8
1965.0,5421,238,10.6
1970.0,16624,1103,6.6
1975.0,22204,2194,6.6
1980.0,32401,731,6.8
1985.0,48625,1183,88.6
1990.0,74203,2528,99.7
1995.0,91859,5894,99.4
2000.0,76562,1090,99.6


For companies registered between 1960 and 1984, only about 7–11% have a sector code, so I cannot use them for the sector questions. The status `Dissolved-20 years` is only found in periods up to 2000–2004. It is a register label for very old closures, not a normal business closure, so it makes the older periods hard to compare with newer ones. Because of this, I use companies registered from 2000 onwards. The 2000–2004 period still has 1,090 of these companies, so in notebook 3 I will check if the results change when I start from 2005 instead.

#### Companies with no registration date
16,192 companies have no registration date. Here I check their type and status, to see what kind of companies they are.

In [13]:
# True for companies with no registration date
no_reg_date = companies["company_reg_date"].isna()

# Their 5 most common types and statuses
display(companies.loc[no_reg_date, "company_type"].value_counts().head(5))
display(companies.loc[no_reg_date, "company_status"].value_counts().head(5))

company_type
Unknown                                 14414
Private                                  1283
Private unlimited                         255
Private limited by shares                 171
Private unlimited with share capital       49
Name: count, dtype: int64

company_status
Dissolved              13899
Dissolved-20 years      2288
Ceased IRL                 2
Ceased                     1
Conv. to IndProvSoc        1
Name: count, dtype: int64

Most companies with no registration date have the type `Unknown` (14,414) or one of the old long labels, and almost all are `Dissolved` or `Dissolved-20 years`. So these are closed companies with old records. Without a registration date I cannot place them in a registration year, so they cannot be used for survival analysis and will be removed when I keep companies registered from 2000 onwards.

#### Dissolution date by status
To measure how long a company survived, I need a closing date. Here I check, for each status, how many companies have a `comp_dissolved_date`.

In [14]:
# Status without extra spaces (only for this comparison)
status = companies["company_status"].str.strip()

# True when the company has a dissolution date
has_dissolved_date = companies["comp_dissolved_date"].notna()

# Number of companies per status, with (True) and without (False) a dissolution date, largest status first
dissolved_by_status = pd.crosstab(status, has_dissolved_date)
dissolved_by_status.loc[status.value_counts().index]

comp_dissolved_date,False,True
company_status,,
Dissolved,1019,457782
Normal,324214,188
Dissolved-20 years,2084,15569
Strike Off Listed,10066,740
Liquidation,9157,54
Ceased IRL,2676,0
Dissolved PostMerger,2100,0
Ceased,1254,0
Struck Off,14,673


Almost all `Dissolved` companies have a dissolution date (457,782), and only 1,019 do not. Most `Normal` companies have no dissolution date, as expected, but 188 have one. `Struck Off` companies mostly have a date. `Dissolved PostMerger`, `Ceased`, `Ceased IRL` and the small statuses never have one. `Strike Off Listed` and `Liquidation` mostly have no date, because these companies are still in the process of closing. Before deciding which statuses count as closed, I need to check what date the statuses without a dissolution date can use.

#### Status date by status
Some closed statuses have no dissolution date. Here I check `company_status_date` (the date the current status started) for each status, and whether it matches the dissolution date when both exist.

In [15]:
# True when the company has a status date
has_status_date = companies["company_status_date"].notna()

# Number of companies per status, with (True) and without (False) a status date
status_date_by_status = pd.crosstab(status, has_status_date)
display(status_date_by_status.loc[status.value_counts().index])

# For companies with both dates: is the status date the same as the dissolution date?
both_dates = has_dissolved_date & has_status_date
same_date = companies["company_status_date"] == companies["comp_dissolved_date"]
display(pd.crosstab(status[both_dates], same_date[both_dates]))

company_status_date,False,True
company_status,,
Dissolved,765,458036
Normal,316976,7426
Dissolved-20 years,2084,15569
Strike Off Listed,0,10806
Liquidation,0,9211
Ceased IRL,0,2676
Dissolved PostMerger,0,2100
Ceased,1100,154
Struck Off,0,687


col_0,False,True
company_status,,
Dissolved,0,457782
Dissolved-20 years,0,15569
Liquidation,54,0
Normal,61,127
Strike Off Listed,740,0
Struck Off,673,0


For `Dissolved` and `Dissolved-20 years` companies, the status date is always the same as the dissolution date. Statuses that have no dissolution date, such as `Dissolved PostMerger` and `Ceased IRL`, do have a status date. So the status date can be used as the closing date for all closed statuses. For `Struck Off`, `Strike Off Listed` and `Liquidation`, the two dates are never the same, so the dissolution date in these rows refers to an earlier event. For these statuses the status date is the date the current status started. `Ceased` companies mostly have no status date (1,100). 7,426 `Normal` companies also have a status date.

#### One row per company
Each company should appear once. Here I check for repeated company numbers and fully repeated rows.

In [16]:
# Rows that are exact copies of another row
print("Fully repeated rows:", companies.duplicated().sum())

# Company numbers that appear more than once
repeated = companies["company_num"].duplicated(keep=False)
print("Rows with a repeated company number:", repeated.sum())

# Those rows, sorted so the pairs are next to each other
columns_to_show = ["company_num", "company_name", "company_status", "company_type",
                   "company_reg_date", "comp_dissolved_date", "company_status_date"]
companies.loc[repeated, columns_to_show].sort_values("company_num")

Fully repeated rows: 0
Rows with a repeated company number: 10


,company_num,company_name,company_status,company_type,company_reg_date,comp_dissolved_date,company_status_date
131509,566202,NAIP MEMOS LIMITED,Normal,LTD - Private Company Limited by Shares,2015-08-07,NaN,2026-04-27
447648,566202,NAIP MEMOS LIMITED,Normal,LTD - Private Company Limited by Shares,2015-08-07,NaN,2024-01-10
607552,566306,MARK DOHERTY ELECTRICAL LIMITED,Normal,LTD - Private Company Limited by Shares,2015-08-10,NaN,NaN
54953,566306,MARK DOHERTY ELECTRICAL LIMITED,Normal,LTD - Private Company Limited by Shares,2015-08-10,NaN,2026-04-27
607551,753407,FEZ DRINKS LIMITED,Normal,LTD - Private Company Limited by Shares,2023-12-06,NaN,2026-05-04
46150,753407,FEZ DRINKS LIMITED,Normal,LTD - Private Company Limited by Shares,2023-12-06,2026-05-11,2026-05-04
163375,758130,JSN HOSPITALITY LIMITED,Normal,LTD - Private Company Limited by Shares,2024-02-20,NaN,NaN
591013,758130,JSN HOSPITALITY LIMITED,Normal,LTD - Private Company Limited by Shares,2024-02-20,NaN,NaN
613287,800043,BRISTOL-MYERS SQUIBB / JOHNSON & JOHNSON EEIG,Normal,European economic interest grouping,2026-10-12,NaN,2026-10-12
685379,800043,BKK ACCOUNT SERVICES LIMITED,Normal,LTD - Private Company Limited by Shares,2025-10-28,NaN,2026-10-12


There are no fully repeated rows, but 5 company numbers appear twice (10 rows). In four pairs it is the same company, with small differences in the dates. One number (800043) belongs to two different companies: a European economic interest grouping and a limited company. The four pairs were registered from 2015 onwards and are still `Normal`. When cleaning, I keep one row for each of the four companies and keep both companies with number 800043, so the company number alone is not a unique key.